In [10]:
import numpy as np
from matplotlib import cm
import matplotlib.pyplot as plt
import glob
from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.interpolate import splev
from scipy.interpolate import splrep
import colorsys
import networkx as nx
import random
import math
import matplotlib.style
import matplotlib as mpl
import pandas as pd
import graph_tool.all as gt
import seaborn as sns
import multiprocessing as mp
import json
import sys
from scipy.stats import hypergeom
from tqdm import tqdm
import networkx.algorithms.community as nx_comm
import disease_module_identification as dmi
import NetworkMetrics as metrics
from scipy import stats
import csv
from collections import defaultdict
from matplotlib.colors import ListedColormap, Normalize
import seaborn as sns
import copy
import random



In [ ]:
path_interactome = "./data/PPI_2022.csv"
G = nx.from_pandas_edgelist(pd.read_csv(path_interactome), 'HGNC_Symbol.1', 'HGNC_Symbol.2')

self_loops = [(u, v) for u, v in G.edges() if u == v]
G.remove_edges_from(self_loops)

connected_components = list(nx.connected_components(G))
lcc = max(len(component) for component in connected_components)

interactome2022 - ppi


In [13]:
path = "./data/Gene_hallmarks.csv"
gene_hallmarks_extended = pd.read_csv(path)
all_hallmarks = gene_hallmarks_extended['aging_mechanisms'].unique()
gene_hallmarks_extended = gene_hallmarks_extended[gene_hallmarks_extended['confidence'] <= 4]

In [14]:
amg_list = list(gene_hallmarks_extended['aging_mechanisms_group'].unique())

In [ ]:
### NODE PERMUTATION TEST
LCC = max(nx.connected_components(G), key=len)
G_sub =  G.subgraph(LCC)
for amg_idx, amg in enumerate(amg_list):
    if amg == 'other':
        continue
    A = gene_hallmarks_extended[gene_hallmarks_extended['aging_mechanisms_group'].isin([amg])]
    B =  set(A[A['GeneId'].isin(G_sub)]['GeneId'])
    lcc_data = metrics.lcc_significance(G_sub, B, degree_preserving='log_binning', n_iter=1000)
    print(amg, len(B),lcc_data['lcc_size'] ,lcc_data['p_val'] , lcc_data['z_score'])

In [ ]:
### EDGE PERMUTATION TEST
LCC = max(nx.connected_components(G), key=len)
G_sub =  G.subgraph(LCC)

rng = np.random.default_rng()
distrbituions = {}
for amg_idx, amg in enumerate(amg_list):
    if amg == 'other':
        continue
    distrbituions[amg] = []

# Step 1: make stub list (node repeated by its degree)
stubs = []
for node, deg in G_sub.degree():
    stubs.extend([node] * deg)

n_permutations = 1000
for i in range(n_permutations):
    # Step 2: shuffle
    rng.shuffle(stubs)

    # Step 3: make new graph
    H = nx.Graph()
    H.add_nodes_from(G_sub.nodes())

    # pair consecutive stubs
    for u, v in zip(stubs[0::2], stubs[1::2]):
        if u != v:  # avoid self-loops
            H.add_edge(u, v)

    for amg_idx, amg in enumerate(amg_list):
        if amg == 'other':
            continue
            
        A = gene_hallmarks_extended[gene_hallmarks_extended['aging_mechanisms_group'].isin([amg])]
        B =  set(A[A['GeneId'].isin(G_sub)]['GeneId'])
        set_a = set(B) & set(G_sub.nodes())
        sub = metrics.extract_lcc(set_a,H)
        distrbituions[amg].append(len(sub))
    
    if i%10 == 0:
        print(f"\rIter {i} de {n_permutations}",end="")
        
print("")

for amg_idx, amg in enumerate(amg_list):
    if amg == 'other':
        continue
    A = gene_hallmarks_extended[gene_hallmarks_extended['aging_mechanisms_group'].isin([amg])]
    B =  set(A[A['GeneId'].isin(G_sub)]['GeneId'])
    set_a = set(B) & set(G_sub.nodes())
    
    lcc = metrics.extract_lcc(set_a,G_sub)
    l_lcc = len(lcc)
    mu = np.mean(distrbituions[amg])
    sigma = np.std(distrbituions[amg])
    z = (l_lcc - mu) / sigma
    pval = 2*(1-stats.norm.cdf(abs(z)))
    print(amg, 'd_mu', mu,'d_sigma', sigma,'z_score', z,'p_val', pval, 'lcc_size', l_lcc)

In [ ]:
path_interactome = "./data/PPI_STRING.csv"
STRING = nx.from_pandas_edgelist(pd.read_csv(path_interactome), 'symbol1', 'symbol2')

self_loops = [(u, v) for u, v in STRING.edges() if u == v]
STRING.remove_edges_from(self_loops)

connected_components = list(nx.connected_components(STRING))
lcc = max(len(component) for component in connected_components)


STRING - ppi


In [ ]:
LCC = max(nx.connected_components(STRING), key=len)
G_sub =  STRING.subgraph(LCC)
for amg_idx, amg in enumerate(amg_list):
    if amg == 'other':
        continue
    A = gene_hallmarks_extended[gene_hallmarks_extended['aging_mechanisms_group'].isin([amg])]
    B =  set(A[A['GeneId'].isin(G_sub)]['GeneId'])
    lcc_data = metrics.lcc_significance(G_sub, B, degree_preserving='log_binning', n_iter=1000)
    print(amg, len(B),lcc_data['lcc_size'] ,lcc_data['p_val'] , lcc_data['z_score'])